# M5 Forecasting Accuracy: Modeling

ใช้ข้อสรุปจาก `eda.ipynb` (Section 7: Feature Summary) และ Validation strategy เดิม

| ขั้น | Train | Forecast | ใช้ทำอะไร |
|---|---|---|---|
| Development | ≤ `d_1913` | `d_1914` – `d_1941` | วัด WRMSSE เทียบกับ baseline |
| Final | ≤ `d_1941` | `d_1942` – `d_1969` | ส่ง Kaggle |

**แนวทาง**
- **LightGBM โมเดลเดียวสำหรับทุกสินค้า** ใช้ loss แบบ Tweedie เพราะ 51–72% ของวันขายได้ 0 (EDA 6.5)
- **พยากรณ์ 28 วันพร้อมกัน (direct)** feature จากยอดขายใช้ข้อมูลเก่ากว่า 28 วันขึ้นไปเท่านั้น
  วันสุดท้ายที่พยากรณ์ก็ยังคำนวณ feature ได้จากยอดขายจริง ไม่ต้องใช้ค่าที่พยากรณ์เอง จึงไม่มี data leakage
- **Train ด้วย 2 ปีล่าสุด** เพราะ RAM จำกัด (16 GB) และข้อมูลช่วงหลังใกล้กับช่วงที่ต้องพยากรณ์มากกว่า
- Hyperparameters ตั้งไว้ล่วงหน้าครั้งเดียว ไม่ได้ปรับจากผลของ validation เพื่อให้คะแนน validation ไม่ถูกปรับให้ดีเกินจริง

**Contents**
1. Setup
2. Build the base table
3. Features
4. WRMSSE
5. Baselines
6. LightGBM (development)
7. Final model and submission
8. Summary

## 1. Setup

In [1]:
import gc

import lightgbm as lgb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

DATA = "m5-data"
TRAIN_END = 1913      # development: train ≤ d_1913, forecast d_1914–d_1941
FINAL_END = 1941      # final: train ≤ d_1941, forecast d_1942–d_1969
HORIZON = 28
HISTORY_DAYS = 730    # จำนวนวันที่ใช้ train
WARMUP_DAYS = 28 + 56 # วันก่อนหน้าที่ต้องมีเพื่อคำนวณ lag / rolling ที่ยาวที่สุด
START_DAY = TRAIN_END - HISTORY_DAYS - WARMUP_DAYS + 1
LAST_DAY = FINAL_END + HORIZON

ID_COLS = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
SEED = 42

## 2. Build the Base Table
ทำแบบเดียวกับ EDA: wide → long, join calendar และ prices แล้วตัดแถวก่อนสินค้าวางขาย
- เพิ่มวัน `d_1942` – `d_1969` (ยอดขายยังไม่รู้) ไว้สำหรับพยากรณ์ final
- `snap` ใช้ค่าของรัฐที่ร้านตั้งอยู่ (EDA 4.3)
- ไม่ใช้ `event_name_2` (EDA 4.1)

In [2]:
sales_wide = pd.read_csv(f"{DATA}/sales_train_evaluation.csv")
calendar = pd.read_csv(f"{DATA}/calendar.csv", parse_dates=["date"])
prices = pd.read_csv(f"{DATA}/sell_prices.csv")
calendar["day"] = calendar["d"].str[2:].astype(int)

future = pd.DataFrame(np.nan, index=sales_wide.index, columns=[f"d_{d}" for d in range(FINAL_END + 1, LAST_DAY + 1)])
day_cols = [f"d_{d}" for d in range(START_DAY, LAST_DAY + 1)]
base = pd.concat([sales_wide[ID_COLS], sales_wide.drop(columns=ID_COLS), future], axis=1)[ID_COLS + day_cols]
base[ID_COLS] = base[ID_COLS].astype("category")

base = base.melt(id_vars=ID_COLS, var_name="d", value_name="sales")
base["day"] = base["d"].str[2:].astype("int16")
base["sales"] = base["sales"].astype("float32")
base = base.drop(columns="d")
del future
base.shape

(26526300, 8)

In [3]:
cal = calendar[["day", "wm_yr_wk", "wday", "month", "event_name_1", "snap_CA", "snap_TX", "snap_WI"]].copy()
cal["mday"] = calendar["date"].dt.day
cal["event_name_1"] = cal["event_name_1"].astype("category")
for c in ["wday", "month", "mday", "snap_CA", "snap_TX", "snap_WI"]:
    cal[c] = cal[c].astype("int8")
cal["day"] = cal["day"].astype("int16")

base = base.merge(cal, on="day", how="left", validate="many_to_one")
base["snap"] = np.select(
    [base["state_id"] == "CA", base["state_id"] == "TX"],
    [base["snap_CA"], base["snap_TX"]],
    default=base["snap_WI"],
).astype("int8")
base = base.drop(columns=["snap_CA", "snap_TX", "snap_WI"])

**Price features** (EDA 5.3: ใช้ข้อเท็จจริงของราคา ไม่นิยามราคาปกติ)
- `sell_price`: ราคาในสัปดาห์นั้น
- `price_chg`: % ที่ราคาเปลี่ยนจากสัปดาห์ก่อน
- `weeks_at_price`: ราคานี้คงที่มากี่สัปดาห์แล้ว
- `days_since_launch`: วันนับจากสัปดาห์แรกที่สินค้ามีราคา (EDA 6.1)

In [4]:
prices = prices.sort_values(["store_id", "item_id", "wm_yr_wk"]).reset_index(drop=True)
by_item = prices.groupby(["store_id", "item_id"])
prices["price_chg"] = (prices["sell_price"] / by_item["sell_price"].shift() - 1).astype("float32")

new_run = prices["price_chg"].fillna(0).ne(0)
run_id = new_run.groupby([prices["store_id"], prices["item_id"]]).cumsum()
prices["weeks_at_price"] = prices.groupby([prices["store_id"], prices["item_id"], run_id]).cumcount().add(1).astype("int16")

week_start = calendar.groupby("wm_yr_wk")["day"].min()
prices["launch_day"] = by_item["wm_yr_wk"].transform("min").map(week_start).astype("int16")

for col in ["store_id", "item_id"]:
    prices[col] = pd.Categorical(prices[col], categories=base[col].cat.categories)
prices["sell_price"] = prices["sell_price"].astype("float32")

base = base.merge(prices, on=["store_id", "item_id", "wm_yr_wk"], how="left", validate="many_to_one")
base = base[base["sell_price"].notna()].reset_index(drop=True)
base["days_since_launch"] = (base["day"] - base["launch_day"]).astype("int16")
base["weeks_at_price"] = base["weeks_at_price"].astype("int16")
base = base.drop(columns=["launch_day", "wm_yr_wk"])
del by_item, new_run, run_id
gc.collect()
base.info()

<class 'pandas.DataFrame'>
RangeIndex: 25721475 entries, 0 to 25721474
Data columns (total 17 columns):
 #   Column             Dtype   
---  ------             -----   
 0   id                 category
 1   item_id            category
 2   dept_id            category
 3   cat_id             category
 4   store_id           category
 5   state_id           category
 6   sales              float32 
 7   day                int16   
 8   wday               int8    
 9   month              int8    
 10  event_name_1       category
 11  mday               int8    
 12  snap               int8    
 13  sell_price         float32 
 14  price_chg          float32 
 15  weeks_at_price     int16   
 16  days_since_launch  int16   
dtypes: category(7), float32(3), int16(3), int8(4)
memory usage: 760.7 MB


## 3. Features
ยอดขายในอดีตทุกตัว **สิ้นสุดเมื่อ 28 วันก่อน** (EDA 6.6)
- `lag_28`, `lag_35`: ยอดขายเมื่อ 28 / 35 วันก่อน (วันเดียวกันของสัปดาห์)
- `rmean_7`, `rmean_28`, `rmean_56`: ค่าเฉลี่ย 7 / 28 / 56 วัน ที่สิ้นสุดเมื่อ 28 วันก่อน
- `rstd_28`: ส่วนเบี่ยงเบนมาตรฐาน 28 วัน
- `wday_mean_4`: ค่าเฉลี่ยของวันเดียวกันของสัปดาห์ 4 สัปดาห์ (lag 28, 35, 42, 49)

แถวของแต่ละสินค้าเรียงตามวัน และไม่มีวันขาดหาย (EDA 5.2) จึง shift ตามแถวได้

In [5]:
by_id = base.groupby("id", observed=True)["sales"]
for lag in [28, 35, 42, 49]:
    base[f"lag_{lag}"] = by_id.shift(lag).astype("float32")
base["wday_mean_4"] = base[["lag_28", "lag_35", "lag_42", "lag_49"]].mean(axis=1, skipna=False).astype("float32")
base = base.drop(columns=["lag_42", "lag_49"])

# rolling ด้วย cumulative sum: sum(t-28-w+1 .. t-28) = cs(t-28) - cs(t-28-w)
cs = by_id.cumsum().groupby(base["id"], observed=True)
cs2 = (base["sales"] ** 2).groupby(base["id"], observed=True).cumsum().groupby(base["id"], observed=True)
for w in [7, 28, 56]:
    base[f"rmean_{w}"] = ((cs.shift(28) - cs.shift(28 + w)) / w).astype("float32")
mean_sq = (cs2.shift(28) - cs2.shift(56)) / 28
base["rstd_28"] = np.sqrt((mean_sq - base["rmean_28"] ** 2).clip(lower=0)).astype("float32")
del by_id, cs, cs2, mean_sq
gc.collect()

CAT_FEATURES = ["item_id", "dept_id", "cat_id", "store_id", "state_id", "event_name_1"]
NUM_FEATURES = [
    "wday", "month", "mday", "snap",
    "sell_price", "price_chg", "weeks_at_price", "days_since_launch",
    "lag_28", "lag_35", "wday_mean_4", "rmean_7", "rmean_28", "rmean_56", "rstd_28",
]
FEATURES = CAT_FEATURES + NUM_FEATURES
base[FEATURES].describe().T[["count", "mean", "min", "max"]]

,count,mean,min,max
wday,25721475.0,3.994073,1.000000,7.000000
month,25721475.0,6.115426,1.000000,12.000000
mday,25721475.0,15.595124,1.000000,31.000000
snap,25721475.0,0.332960,0.000000,1.000000
sell_price,25721475.0,4.463880,0.010000,48.779999
price_chg,25692306.0,0.001204,-0.997988,496.000000
weeks_at_price,25721475.0,93.019786,1.000000,282.000000
days_since_launch,25721475.0,1170.642932,0.000000,1968.000000
lag_28,24867755.0,1.279407,0.000000,606.000000
lag_35,24654325.0,1.278168,0.000000,606.000000


## 4. WRMSSE
ตัววัดของการแข่งขัน คำนวณครบ 12 ระดับ (ตั้งแต่ยอดรวมทั้งหมดไปจนถึงสินค้า × ร้าน รวม 42,840 series)
- **RMSSE** ของแต่ละ series = RMSE ของการพยากรณ์ / RMSE ของการพยากรณ์ว่า "พรุ่งนี้ = วันนี้" ในช่วง train
  (นับตั้งแต่วันแรกที่ series นั้นขายได้)
- **น้ำหนัก** = ยอดขายเป็นเงินใน 28 วันสุดท้ายของช่วง train (หน่วย × ราคา) ในแต่ละระดับรวมเป็น 1
- **WRMSSE** = ค่าเฉลี่ยของ 12 ระดับ (ยิ่งต่ำยิ่งดี)

In [6]:
LEVELS = {
    "1 total": [], "2 state": ["state_id"], "3 store": ["store_id"],
    "4 category": ["cat_id"], "5 department": ["dept_id"],
    "6 state × category": ["state_id", "cat_id"], "7 state × department": ["state_id", "dept_id"],
    "8 store × category": ["store_id", "cat_id"], "9 store × department": ["store_id", "dept_id"],
    "10 item": ["item_id"], "11 item × state": ["item_id", "state_id"], "12 item × store": ["item_id", "store_id"],
}


class WRMSSE:
    def __init__(self, sales_wide, prices, calendar, train_end):
        self.ids = sales_wide[ID_COLS].astype(str).reset_index(drop=True)
        train = sales_wide[[f"d_{d}" for d in range(1, train_end + 1)]].to_numpy(np.float64)

        last_days = range(train_end - HORIZON + 1, train_end + 1)
        weeks = calendar.set_index("day").loc[list(last_days), "wm_yr_wk"].to_numpy()
        price_wide = (
            prices.assign(store_id=prices["store_id"].astype(str), item_id=prices["item_id"].astype(str))
            .pivot(index=["store_id", "item_id"], columns="wm_yr_wk", values="sell_price")
            .reindex(pd.MultiIndex.from_frame(self.ids[["store_id", "item_id"]]))[weeks]
        )
        dollars = (train[:, -HORIZON:] * price_wide.fillna(0).to_numpy()).sum(axis=1)

        self.levels = {}
        for name, cols in LEVELS.items():
            key = self.ids.groupby(cols, sort=True).ngroup().to_numpy() if cols else None
            agg = self._agg(train, key)
            diffs = np.diff(agg, axis=1) ** 2
            started = np.cumsum(agg[:, :-1] > 0, axis=1) > 0  # นับตั้งแต่วันแรกที่ขายได้
            scale = (diffs * started).sum(axis=1) / started.sum(axis=1)
            weight = self._agg(dollars[:, None], key)[:, 0]
            self.levels[name] = (key, scale, weight / weight.sum())

    @staticmethod
    def _agg(values, key):
        if key is None:
            return values.sum(axis=0, keepdims=True)
        return pd.DataFrame(values).groupby(key).sum().to_numpy()

    def score(self, pred, actual):
        """pred, actual: arrays (30,490 × 28) เรียงแถวเดียวกับ sales_wide"""
        per_level = {}
        for name, (key, scale, weight) in self.levels.items():
            err = ((self._agg(actual, key) - self._agg(pred, key)) ** 2).mean(axis=1)
            per_level[name] = np.nansum(weight * np.sqrt(err / scale))
        per_level = pd.Series(per_level, name="WRMSSE")
        return per_level.mean(), per_level


evaluator = WRMSSE(sales_wide, prices, calendar, TRAIN_END)
valid_cols = [f"d_{d}" for d in range(TRAIN_END + 1, TRAIN_END + HORIZON + 1)]
valid_actual = sales_wide[valid_cols].to_numpy(np.float64)

## 5. Baselines
ใช้เป็นเกณฑ์ว่าโมเดลดีกว่าวิธีง่าย ๆ แค่ไหน
- **Mean 28 days:** ทุกวันใช้ค่าเฉลี่ย 28 วันสุดท้ายของช่วง train
- **Seasonal naive:** ทำซ้ำยอดขาย 7 วันสุดท้าย 4 รอบ (benchmark ทางการของ M5)

In [7]:
train_tail = sales_wide[[f"d_{d}" for d in range(TRAIN_END - HORIZON + 1, TRAIN_END + 1)]].to_numpy(np.float64)
baselines = {
    "Mean 28 days": np.repeat(train_tail.mean(axis=1, keepdims=True), HORIZON, axis=1),
    "Seasonal naive": np.tile(train_tail[:, -7:], HORIZON // 7),
}
results = {name: evaluator.score(pred, valid_actual) for name, pred in baselines.items()}
pd.Series({name: r[0] for name, r in results.items()}, name="WRMSSE").to_frame().style.format("{:.4f}")

,WRMSSE
Mean 28 days,1.0823
Seasonal naive,0.8697


## 6. LightGBM (Development)
Train ≤ `d_1913` (2 ปีล่าสุด), พยากรณ์ `d_1914` – `d_1941`
ชุด `d_1914` – `d_1941` ใช้ดู loss ระหว่าง train เท่านั้น ไม่ได้ใช้หยุด train (no early stopping)

In [8]:
PARAMS = {
    "objective": "tweedie",
    "tweedie_variance_power": 1.1,
    "learning_rate": 0.05,
    "num_leaves": 127,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 0.1,
    "max_bin": 255,
    "num_threads": 8,
    "seed": SEED,
    "verbose": -1,
}
NUM_ROUNDS = 1000


def train_model(train_end, valid=True):
    train_mask = base["day"].between(train_end - HISTORY_DAYS + 1, train_end)
    train_set = lgb.Dataset(base.loc[train_mask, FEATURES], base.loc[train_mask, "sales"],
                            categorical_feature=CAT_FEATURES, free_raw_data=True)
    valid_sets, callbacks = [], [lgb.log_evaluation(100)]
    if valid:
        valid_mask = base["day"].between(train_end + 1, train_end + HORIZON)
        valid_sets = [lgb.Dataset(base.loc[valid_mask, FEATURES], base.loc[valid_mask, "sales"], reference=train_set)]
    print(f"Training rows: {train_mask.sum():,}")
    return lgb.train(PARAMS, train_set, NUM_ROUNDS, valid_sets=valid_sets, callbacks=callbacks)


def predict_wide(model, first_day):
    """พยากรณ์ 28 วันตั้งแต่ first_day แล้วจัดเป็น wide (30,490 × 28) เรียงตาม sales_wide
    สินค้าที่ยังไม่วางขาย (ไม่มีแถว) = 0"""
    mask = base["day"].between(first_day, first_day + HORIZON - 1)
    pred = base.loc[mask, ["id", "day"]].assign(pred=model.predict(base.loc[mask, FEATURES]))
    wide = pred.pivot_table(index="id", columns="day", values="pred", observed=True)
    return wide.reindex(index=sales_wide["id"], columns=range(first_day, first_day + HORIZON)).fillna(0).to_numpy()

In [9]:
%%time
dev_model = train_model(TRAIN_END)
dev_pred = predict_wide(dev_model, TRAIN_END + 1)
results["LightGBM"] = evaluator.score(dev_pred, valid_actual)

def results_table(results):
    table = pd.DataFrame({name: r[1] for name, r in results.items()})
    overall = pd.DataFrame({name: [r[0]] for name, r in results.items()}, index=["WRMSSE (mean of 12)"])
    return pd.concat([overall, table]).style.format("{:.4f}")

results_table(results)

Training rows: 21,736,144


[100]	valid_0's tweedie: 14.4364


[200]	valid_0's tweedie: 14.4353


[300]	valid_0's tweedie: 14.4381


[400]	valid_0's tweedie: 14.4407


...

[800]	valid_0's tweedie: 14.4452


[900]	valid_0's tweedie: 14.4464


[1000]	valid_0's tweedie: 14.4481


CPU times: user 1h 23min 40s, sys: 2min, total: 1h 25min 41s
Wall time: 16min 55s


,Mean 28 days,Seasonal naive,LightGBM
WRMSSE (mean of 12),1.0823,0.8697,0.6407
1 total,1.1571,0.7514,0.5311
2 state,1.1311,0.7544,0.5359
3 store,1.1224,0.7993,0.5905
4 category,1.1527,0.7620,0.5327
5 department,1.2112,0.8171,0.5579
6 state × category,1.1258,0.7830,0.5569
7 state × department,1.1640,0.8304,0.5844
8 store × category,1.0983,0.8393,0.6244
9 store × department,1.1140,0.8919,0.6690


**Total sales: forecast vs actual**
ยอดขายรวมทุกสินค้ารายวัน 8 สัปดาห์สุดท้ายของช่วง train และ 4 สัปดาห์ที่พยากรณ์

In [10]:
history_days = list(range(TRAIN_END - 55, TRAIN_END + HORIZON + 1))
actual_total = sales_wide[[f"d_{d}" for d in history_days]].sum()
dates = calendar.set_index("day").loc[history_days, "date"]
forecast_dates = dates.loc[TRAIN_END + 1:]

fig = go.Figure([
    go.Scatter(x=dates, y=actual_total.to_numpy(), name="Actual", line=dict(color="#52514e", width=2)),
    go.Scatter(x=forecast_dates, y=dev_pred.sum(axis=0), name="LightGBM", line=dict(color="#2a78d6", width=2)),
    go.Scatter(x=forecast_dates, y=baselines["Seasonal naive"].sum(axis=0), name="Seasonal naive",
               line=dict(color="#eb6834", width=2, dash="dot")),
])
fig.add_vline(x=dates.loc[TRAIN_END + 1], line_dash="dot", line_color="gray")
fig.update_layout(title="Total daily units: forecast vs actual (d_1914 – d_1941)",
                  yaxis_title="Units sold per day", hovermode="x unified", plot_bgcolor="white")
fig.show()

**Feature importance** (gain = loss ที่ลดลงรวมจากการ split ด้วย feature นั้น)

In [11]:
importance = (
    pd.Series(dev_model.feature_importance("gain"), index=FEATURES, name="gain")
    .pipe(lambda s: s / s.sum()).sort_values().reset_index().rename(columns={"index": "feature"})
)
fig = px.bar(importance, x="gain", y="feature", orientation="h",
             color_discrete_sequence=["#2a78d6"], labels={"gain": "Share of total gain", "feature": ""},
             title="Feature importance (development model)", height=550)
fig.update_xaxes(tickformat=".0%")
fig.show()

## 7. Final Model and Submission
Train ใหม่ด้วยข้อมูลถึง `d_1941` (2 ปีล่าสุด) ด้วย parameters ชุดเดิม แล้วพยากรณ์ `d_1942` – `d_1969`
- แถว `_evaluation` = ค่าพยากรณ์ `d_1942` – `d_1969`
- แถว `_validation` = ยอดขายจริง `d_1914` – `d_1941` (ข้อมูลช่วงนี้เปิดเผยแล้วในไฟล์ evaluation)

In [12]:
%%time
del dev_model
gc.collect()
final_model = train_model(FINAL_END, valid=False)
final_pred = predict_wide(final_model, FINAL_END + 1)

Training rows: 21,808,727


CPU times: user 1h 19min 47s, sys: 1min 31s, total: 1h 21min 19s
Wall time: 13min 51s


In [13]:
f_cols = [f"F{i}" for i in range(1, HORIZON + 1)]
evaluation = pd.DataFrame(final_pred, columns=f_cols).assign(id=sales_wide["id"].to_numpy())
validation = pd.DataFrame(valid_actual, columns=f_cols).assign(
    id=sales_wide["id"].str.replace("_evaluation", "_validation").to_numpy()
)
sample = pd.read_csv(f"{DATA}/sample_submission.csv", usecols=["id"])
submission = sample.merge(pd.concat([validation, evaluation]), on="id", how="left")
assert submission[f_cols].notna().all().all() and len(submission) == len(sample)

submission.to_csv("submission.csv", index=False)
print(f"Saved submission.csv: {submission.shape}")
submission.head()

Saved submission.csv: (60980, 29)


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,HOBBIES_1_001_CA_1_validation,0.0,0.0,0.0,2.0,0.0,3.0,5.0,0.0,0.0,...,2.0,4.0,0.0,0.0,0.0,0.0,3.0,3.0,0.0,1.0
1,HOBBIES_1_002_CA_1_validation,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,2.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
2,HOBBIES_1_003_CA_1_validation,0.0,0.0,1.0,1.0,0.0,2.0,1.0,0.0,0.0,...,1.0,0.0,2.0,0.0,0.0,0.0,2.0,3.0,0.0,1.0
3,HOBBIES_1_004_CA_1_validation,0.0,0.0,1.0,2.0,4.0,1.0,6.0,4.0,0.0,...,1.0,1.0,0.0,4.0,0.0,1.0,3.0,0.0,2.0,6.0
4,HOBBIES_1_005_CA_1_validation,1.0,0.0,2.0,3.0,1.0,0.0,3.0,2.0,3.0,...,0.0,0.0,0.0,2.0,1.0,0.0,0.0,2.0,1.0,0.0


## 8. Summary

**ผล development (`d_1914` – `d_1941`)**

| Model | WRMSSE |
|---|---|
| Mean 28 days | 1.0823 |
| Seasonal naive (benchmark M5) | 0.8697 |
| **LightGBM** | **0.6407** |

- LightGBM ดีกว่า seasonal naive 26% และดีกว่าในทุกระดับทั้ง 12 ระดับ
- ระดับรวม (ยอดรวม, รัฐ, หมวด) ได้ 0.53–0.56 ส่วนระดับสินค้าได้ 0.83–0.84 เพราะยอดขายรายสินค้ามีวันที่ขายได้ 0 มากจนพยากรณ์ยาก
- ยอดรวม 28 วันพยากรณ์ได้ 1.155 ล้านชิ้น ขณะที่ยอดขายจริงคือ 1.232 ล้านชิ้น **ต่ำกว่าจริง 6.2%**

**Feature importance**
- `rmean_56` คิดเป็น 61% ของ gain ทั้งหมด ตามด้วย `rmean_28`, `item_id`, `rmean_7` (ราว 9–10% ต่อตัว)
  ตรงกับ EDA 6.6 ที่พบว่าค่าเฉลี่ยหลายวันสัมพันธ์กับยอดขายดีกว่ายอดขายวันเดียว
- `snap` และ `price_chg` ได้ gain ไม่ถึง 0.1% ส่วนหนึ่งเพราะ gain เป็นค่ารวมทุกสินค้า
  ผลของ SNAP มีเฉพาะ FOODS และราคาเปลี่ยนไม่บ่อย (EDA 5.1, 6.4) ยังไม่ได้ทดสอบว่าถ้าตัดออกแล้วคะแนนเปลี่ยนไหม

**Final**
- Train ใหม่ถึง `d_1941` แล้วพยากรณ์ `d_1942` – `d_1969` บันทึกเป็น `submission.csv`
  (60,980 แถว ไม่มีค่าว่างหรือค่าติดลบ) ค่าพยากรณ์มีรูปแบบรายสัปดาห์ เสาร์–อาทิตย์สูงสุดตาม EDA 6.2

**ข้อจำกัดและสิ่งที่ควรลองต่อ**
- Validation tweedie loss ต่ำสุดราว round 200 แล้วค่อย ๆ สูงขึ้น แต่ไม่ได้ลดจำนวน round
  เพราะการเลือกจากชุดเดียวกับที่ใช้วัดผลจะทำให้คะแนนดีเกินจริง ถ้าจะปรับ ควรแยก validation อีกชุดไว้สำหรับเลือก (เช่น `d_1886` – `d_1913`)
- ใช้ validation แค่ 28 วันชุดเดียว ควรทำ walk-forward หลายช่วงตามที่เขียนไว้ใน Validation strategy
- ยอดรวมพยากรณ์ต่ำกว่าจริง 6.2% อาจลองปรับ `tweedie_variance_power` หรือปรับระดับยอดรวม
- ลองพยากรณ์ทีละวัน (recursive) เพื่อใช้ lag < 28 (EDA 6.6)
